# Running the campaign, one batch at a time

This notebook executes one batch of the IBM TN-VQE campaign through Cebule's
`TN_QC_OPT` task. It is deliberately one batch per session: each batch maps
to a separate IBM access-plan purchase, and mixing them makes the spend
against a given plan impossible to read afterwards.

The campaign itself, and the reasoning behind what each run does, is in
[the campaign README](README.md). The batch files are generated by
[`ibm_tn_vqe_campaign.ipynb`](ibm_tn_vqe_campaign.ipynb) and are inputs
here, never rewritten.

**Three things protect purchased time.** Nothing submits until `SUBMIT` is
set to `True`, so a stray "Run All" costs nothing. Every completed run is
checkpointed, so re-running the notebook resumes rather than re-spending.
And the loop stops if cumulative estimated spend would exceed the batch's
plan budget.

Credentials are read from the environment and never entered here. Set
`CEBULE_EMAIL` and `CEBULE_PASSWORD` in the shell that launches Jupyter.

In [ ]:
BATCH = "batch0_classical_only"   # batch0 needs no QPU time and no IBM plan
SUBMIT = False                    # set True to actually submit
STOP_AFTER = None                 # e.g. 1 to submit a single run, then stop

# Batch order is deliberate: batch0 is classical, batch1 is the cheapest
# hardware work and validates the pipeline before purchased time is used.
BATCHES = ["batch0_classical_only", "batch1_open_plan",
           "batch2_flex_plan", "batch3_premium_plan"]
PLAN_BUDGET_MIN = {"batch0_classical_only": None, "batch1_open_plan": 10,
                   "batch2_flex_plan": 400, "batch3_premium_plan": 5200}

assert BATCH in BATCHES, f"unknown batch {BATCH!r}"
print(f"batch:  {BATCH}")
print(f"submit: {SUBMIT}" + ("" if SUBMIT else "   (dry run: inputs are built and shown, nothing is sent)"))

In [ ]:
import csv
import hashlib
import json
import os
import pathlib
import sys
import time

CAMPAIGN = pathlib.Path.cwd()
while not (CAMPAIGN / "pyproject.toml").exists():
    CAMPAIGN = CAMPAIGN.parent
REPO = CAMPAIGN
CAMPAIGN = REPO / "data" / "benchmarks" / "ibm_tn-vqe_qesem"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "utils"))

runs = list(csv.DictReader((CAMPAIGN / f"{BATCH}.csv").open()))
budget_min = PLAN_BUDGET_MIN[BATCH]
estimated_min = sum(float(r.get("Est_QPU_Time_S") or 0) for r in runs) / 60

print(f"{len(runs)} runs")
print(f"estimated QPU time: {estimated_min:,.2f} min"
      + (f" of {budget_min} min plan budget" if budget_min else "  (no QPU time: classical only)"))

## Where results go

Each finished run is appended to an NDJSON file as a single line: the run's
identifying columns, the returned energy, the cost history, and the
wall-clock time it took. Wall-clock is recorded because the campaign
measures whether TN-VQE displaces quantum work onto CPU, and `TNQCOptResult`
returns no timing of its own.

The file is the checkpoint. A run whose `Case_ID` already appears in it is
skipped, so an interrupted batch resumes where it stopped.

In [ ]:
RESULTS = CAMPAIGN / "results" / f"{BATCH}.ndjson"
RESULTS.parent.mkdir(parents=True, exist_ok=True)

def completed_case_ids() -> set[str]:
    if not RESULTS.exists():
        return set()
    with RESULTS.open() as f:
        return {json.loads(line)["Case_ID"] for line in f if line.strip()}

done = completed_case_ids()
pending = [r for r in runs if r["Case_ID"] not in done]
print(f"{len(done)} completed, {len(pending)} pending -> {RESULTS.relative_to(REPO)}")

## Building one run's input

Three things are assembled per run, and each is checked rather than assumed.

The **circuit** comes from the committed OpenQASM file the run names, and its
SHA-256 is verified against the value recorded in the campaign. A mismatch
means the file changed after the campaign was generated, which would make
the run incomparable with the rest, so it raises rather than proceeding.
Supplying `qasm_ansatz` also changes `n_layers_circuit`'s effective default
from 3 to 1 upstream, so it is passed explicitly.

The **initial circuit parameters** are the campaign's pinned draw, keyed on
the ansatz alone so that runs sharing a circuit start from the same point.

The **Hamiltonian** is built locally for Jordan-Wigner runs, from the run's
own geometry, basis and active space. Two classes cannot be built here and
are reported and skipped rather than silently mis-built: mol_map runs, whose
constraint encoding is Cebule's and needs a MOL_MAP task to produce the
operator, and q-vSZP runs, since that basis is not one PySCF knows and this
repository parses only its shell structure, not the exponents a calculation
would need.

One mapping is worth stating, because the campaign's columns do not spell
it out. A plain-VQE run is `TN_QC_OPT` with the tensor network switched
off: `optimization_mode="circuit"` freezes `θ`, and zero network layers
means there is no `θ` to freeze. That is why `TN_Layers_Network` and
`TN_Ansatz` are empty on those runs, and why the three methods remain
comparable, since all three reach the QPU through the same task.

In [ ]:
import numpy as np
from _ansatz_builders import PHI_INIT_SEED

def verified_qasm(run: dict) -> str:
    path = REPO / run["Qasm_Ansatz_File"]
    text = path.read_text()
    digest = hashlib.sha256(text.encode()).hexdigest()[:12]
    if digest != run["Qasm_Ansatz_SHA256"]:
        raise ValueError(
            f"{path.name}: sha256 {digest} does not match the campaign's "
            f"{run['Qasm_Ansatz_SHA256']}. The pinned circuit changed after "
            f"the campaign was generated; regenerate or restore it before running."
        )
    return text

def phi_init(run: dict) -> list[float]:
    n = int(run["Num_Opt_Params_Phi"])
    return (2 * np.pi * np.random.default_rng(PHI_INIT_SEED).random(n)).tolist()

def parse_geometry(spec: str) -> list[tuple[str, tuple[float, float, float]]]:
    atoms = []
    for atom in spec.split(";"):
        symbol, x, y, z = atom.split()
        atoms.append((symbol, (float(x), float(y), float(z))))
    return atoms

from qpubench.schemas.observable import SparsePauliObservable

def to_cebule_operators(observable) -> tuple[list[float], list[str]]:
    """(coefficients, "X0 Y1 Z3" token strings), the inverse of
    SparsePauliObservable.from_cebule_operators. The identity term encodes
    as the empty string, which is what that parser reads it back from."""
    # A molecular Hamiltonian is Hermitian, so every coefficient is real;
    # Cebule's h_coeff_values is a list[float], so assert rather than discard.
    assert all(abs(term.coefficient.im) < 1e-12 for term in observable.terms), \
        "complex coefficient in a Hamiltonian Cebule expects to be real"
    coefficients = [term.coefficient.re for term in observable.terms]
    operators = [
        # op.value, not op: PauliLabel subclasses str but is a plain Enum,
        # so an f-string renders it "PauliLabel.X" rather than "X".
        " ".join(f"{op.value}{qubit}"
                 for qubit, op in zip(term.qubit_indices, term.pauli_ops))
        for term in observable.terms
    ]
    round_trip = SparsePauliObservable.from_cebule_operators(
        operators, coefficients, observable.num_qubits)
    assert round_trip.terms == observable.terms, "Cebule operator encoding is lossy"
    return coefficients, operators

def unbuildable_reason(run: dict) -> str | None:
    """Why this run's Hamiltonian cannot be built locally, or None if it can."""
    if run["Mapper"] != "JW":
        return "mol_map encoding is Cebule's, so the operator needs a MOL_MAP task"
    if run["Basis"] == "qvSZP":
        return ("q-vSZP is not a PySCF basis, and hamiltonian_sources/qvszp.py "
                "parses shell letters and function counts only, not exponents")
    return None

def hamiltonian(run: dict):
    """(coefficients, operators) for the run, or None if it cannot be built here."""
    if unbuildable_reason(run) is not None:
        return None
    from qpubench.hamiltonian_sources.ab_initio import build_qubit_hamiltonian
    restricted = run["Active_Space"] != "full"
    observable, _ = build_qubit_hamiltonian(
        parse_geometry(run["Geometry"]),
        basis=run["Basis"],
        charge=int(run["Charge"]),
        multiplicity=int(run["Multiplicity"]),
        active_electrons=int(run["Active_Electrons"]) if restricted else None,
        active_orbitals=int(run["Active_Orbitals"]) if restricted else None,
        mapper="jordan_wigner",
    )
    return to_cebule_operators(observable)

In [ ]:
from qpubench.schemas.mirrors.mqsdk_cebule import TNAnsatz, TNQCOptInput

def build_input(run: dict) -> TNQCOptInput | None:
    h = hamiltonian(run)
    if h is None:
        return None
    coefficients, operators = h

    # A plain-VQE run is TN_QC_OPT with the network switched off: mode
    # "circuit" freezes theta, and zero network layers means there is no
    # theta to freeze.  tn_theta_parameter_count validates n_layers_network
    # >= 0 explicitly and returns 0 parameters at 0, which is why the
    # campaign leaves TN_Layers_Network and TN_Ansatz empty on these runs.
    plain_vqe = run["Method"] == "VQE"
    network_only = run["Optimization_Mode"] == "network"

    return TNQCOptInput(
        h_coeff_values=coefficients,
        h_operators=operators,
        n_iterations=int(run["Iterations"]),
        n_layers_network=0 if plain_vqe else int(run["TN_Layers_Network"]),
        qasm_ansatz=verified_qasm(run),
        # Supplying qasm_ansatz makes 1 the effective default upstream; pin it.
        n_layers_circuit=1,
        # Irrelevant at zero layers, so the model's own default stands.
        tn_ansatz=TNAnsatz.ROTATION_3PARAM if plain_vqe else TNAnsatz(run["TN_Ansatz"]),
        theta_init=[],
        phi_init=phi_init(run),
        opt_method=run["Optimizer"],
        opt_options=json.loads(run["Opt_Options"]),
        n_shots=None if network_only else int(run["Shots"]),
        backend=run["Backend_Platform"],
        measurement_method="pauli" if network_only else run["Measurement_Method"],
        optimization_mode="circuit" if plain_vqe else run["Optimization_Mode"],
    )

# Build the first pending run to see what a submission looks like.
if pending:
    sample = build_input(pending[0])
    if sample is None:
        print(f"run {pending[0]['Case_ID']}: cannot be built here -- "
              f"{unbuildable_reason(pending[0])}")
    else:
        print(f"run {pending[0]['Case_ID']}: {pending[0]['Molecule']}/{pending[0]['Basis']}, "
              f"{pending[0]['Method']}, {pending[0]['Ansatz']}")
        print(f"  {len(sample.h_coeff_values)} Hamiltonian terms, "
              f"{len(sample.phi_init)} circuit parameters, "
              f"{sample.n_iterations} evaluations, backend {sample.backend}")

## Submitting

The session is opened once and reused. Credentials come from the
environment: this cell reads `CEBULE_EMAIL` and `CEBULE_PASSWORD` and does
not prompt for them, so nothing is typed into a notebook that gets saved to
disk.

In [ ]:
session = None
if SUBMIT:
    import mqsdk
    email, password = os.environ.get("CEBULE_EMAIL"), os.environ.get("CEBULE_PASSWORD")
    if not (email and password):
        raise RuntimeError(
            "Set CEBULE_EMAIL and CEBULE_PASSWORD in the shell that launched "
            "Jupyter. They are deliberately not read from this notebook."
        )
    session = mqsdk.Cebule(email, password)
    print("session opened")
else:
    print("SUBMIT is False: the loop below will build and validate, not send")

## The run loop

One run at a time, checkpointed after each. The spend guard is on estimated
time rather than billed, since billed is only known after the fact; it is
there to stop a runaway loop, not to account precisely.

In [ ]:
from qpubench.schemas.mirrors.mqsdk_cebule import CebuleTaskType, TNQCOptResult

spent_min = sum(float(r["Est_QPU_Time_S"] or 0) for r in runs if r["Case_ID"] in done) / 60
submitted = 0

for run in pending:
    case, estimate_min = run["Case_ID"], float(run.get("Est_QPU_Time_S") or 0) / 60

    if budget_min is not None and spent_min + estimate_min > budget_min:
        print(f"stopping before run {case}: would take the batch to "
              f"{spent_min + estimate_min:,.1f} min of a {budget_min} min budget")
        break
    if STOP_AFTER is not None and submitted >= STOP_AFTER:
        print(f"stopping after {submitted} run(s), as STOP_AFTER asked")
        break

    task_input = build_input(run)
    if task_input is None:
        print(f"  skip {case}: {unbuildable_reason(run)}")
        continue

    label = (f"{run['Molecule']}/{run['Basis']} {run['Method']} "
             f"{run['Ansatz']} {run['Optimization_Mode']}")
    if not SUBMIT:
        print(f"  would submit {case}: {label}, {estimate_min:.2f} min")
        continue

    print(f"  submitting {case}: {label} ...", end=" ", flush=True)
    started = time.time()
    task = session.cebule.create_task(
        f"{BATCH}-case{case}", CebuleTaskType.TN_QC_OPT,
        **task_input.model_dump(exclude={"task_type"}, exclude_none=True),
    )
    result = TNQCOptResult.from_task_result(task.result())
    wall_s = time.time() - started

    with RESULTS.open("a") as f:
        f.write(json.dumps({
            "Case_ID": case, "Batch": BATCH,
            "Molecule": run["Molecule"], "Basis": run["Basis"],
            "Mapper": run["Mapper"], "Method": run["Method"],
            "Ansatz": run["Ansatz"], "Optimization_Mode": run["Optimization_Mode"],
            "Measurement_Method": run["Measurement_Method"],
            "vqe_energy": result.vqe_energy,
            "function_calls": result.function_calls,
            "cost_history": result.cost_history,
            "wall_clock_s": round(wall_s, 3),
            "estimated_qpu_s": float(run.get("Est_QPU_Time_S") or 0),
        }) + "\n")

    spent_min += estimate_min
    submitted += 1
    print(f"E = {result.vqe_energy:.6f} Ha, {wall_s:.1f} s wall clock")

print(f"\n{submitted} submitted this session, {len(completed_case_ids())} of {len(runs)} complete")

## Where the batch stands

Wall-clock time is summarised alongside the energies because it is the
classical half of the measurement: a `network` run consumes no QPU time at
all, so its wall clock is pure CPU cost, and comparing it against the
`both`-mode runs it controls is what quantifies the trade the method makes.

In [ ]:
if RESULTS.exists():
    records = [json.loads(line) for line in RESULTS.open() if line.strip()]
    print(f"{len(records)} of {len(runs)} runs complete\n")
    print(f"{'case':>5}  {'molecule':16}{'method':9}{'mode':9}{'energy (Ha)':>13}{'wall (s)':>10}")
    for rec in sorted(records, key=lambda r: int(r["Case_ID"])):
        print(f"{rec['Case_ID']:>5}  {rec['Molecule'] + '/' + rec['Basis']:16}"
              f"{rec['Method']:9}{rec['Optimization_Mode']:9}"
              f"{rec['vqe_energy']:>13.6f}{rec['wall_clock_s']:>10.1f}")
    total_wall = sum(r["wall_clock_s"] for r in records)
    print(f"\ntotal wall clock {total_wall / 60:,.1f} min, "
          f"estimated QPU {sum(r['estimated_qpu_s'] for r in records) / 60:,.1f} min")
else:
    print("nothing completed yet")

## Moving to the next batch

Set `BATCH` to the next entry and re-run. Each is a separate purchase, so
the order matters: `batch0` costs nothing, `batch1` fits the Open Plan's
free quota and exists to prove the pipeline works, and only then is
purchased time committed to `batch2` and `batch3`.

Two things still need settling before the mol_map runs can execute. Their
Hamiltonians need a MOL_MAP task, and their measurement counts are assumed
lower bounds rather than measured, which is what the campaign README lists
as the first thing a real execution should report back.